# DS604 Introduction to Data Management, Autumn 2026
## Milestone 4: Executing Analytical Queries on PostgreSQL
**Group members:** Pratham Patel (202618057), Rohan Deshpande (202618058), Hiteshsinh Rathod (202618040)

**Server:** PostgreSQL at 10.100.71.21

Each query from the Milestone 1 list (Q01 to Q24) is written in SQL, sent to the server with `pandas.read_sql_query`, collected in a DataFrame, and the first 10 rows are printed with `head(10)`.

**Revenue** is defined throughout as `SUM(qty * price)` over `order_items` (as in Milestone 1). `payments.amount` is analysed separately (Q16).

**Data-quality facts from Milestone 2 that affect the results**
- 40,767 orders have no `order_items`, so item-based revenue queries only cover the other 259,233 orders.
- Every order has a promotion, so the "no promotion" group in Q10 is expected to be empty.
- 120,020 orders (40%) are dated before the customer's `signup_date`, so Q24 will show negative day gaps.
- `payments.amount` is unrelated to item totals, so Q16 shows large differences.

## 1. Imports

In [ ]:
import getpass
import warnings
from decimal import Decimal
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
print("pandas", pd.__version__)

## 2. Connect to the PostgreSQL Server


In [ ]:
DB_HOST = "10.100.71.21"
DB_PORT = 5432
DB_NAME = input("Database name: ").strip()
DB_USER = input("User name: ").strip()
DB_PASS = getpass.getpass("Password: ")

url = URL.create("postgresql+psycopg2", username=DB_USER, password=DB_PASS,
                 host=DB_HOST, port=DB_PORT, database=DB_NAME)
engine = create_engine(url, pool_pre_ping=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())

## 3. Helper Function


In [ ]:
def run_query(sql):
    try:
        with engine.connect() as conn:
            df = pd.read_sql_query(text(sql), conn)
    except Exception as err:
        print("SQLAlchemy path failed (" + type(err).__name__ + "), retrying with raw connection")
        raw = engine.raw_connection()
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                df = pd.read_sql_query(sql, raw)
        finally:
            raw.close()
    for c in df.columns:
        if df[c].dtype == object and len(df) and isinstance(df[c].iloc[0], Decimal):
            df[c] = df[c].astype(float)
    print(f"Rows returned: {len(df):,}")
    return df

## 4. Sanity Check: Row Counts


In [ ]:
tables = ["categories","suppliers","stores","promotions","customers","employees",
          "products","orders","order_items","payments","shipments","returns"]
sql = " UNION ALL ".join(f"SELECT '{t}' AS table_name, COUNT(*) AS n_rows FROM {t}" for t in tables)
df = run_query(sql)
df.head(10)

## 5. Analytical Queries (Q01 to Q24)

### Q01. Customers by city
**Question:** Count customers in each city and show the percentage of total customers.

**SQL concepts:** GROUP BY, COUNT, window function for percentage

In [ ]:
sql_q01 = """
SELECT city,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM customers
GROUP BY city
ORDER BY customers DESC;
"""
df_q01 = run_query(sql_q01)
df_q01.head(10)

### Q02. Customer sign-up trend
**Question:** Count new customers per year and month using signup_date.

**SQL concepts:** Date functions, GROUP BY

In [ ]:
sql_q02 = """
SELECT EXTRACT(YEAR FROM signup_date)  AS signup_year,
       EXTRACT(MONTH FROM signup_date) AS signup_month,
       COUNT(*) AS new_customers
FROM customers
GROUP BY 1, 2
ORDER BY 1, 2;
"""
df_q02 = run_query(sql_q02)
df_q02.head(10)

### Q03. Sales by store
**Question:** For each store (and its city) calculate the number of orders and total revenue, ordered by revenue.

**SQL concepts:** JOIN, SUM, GROUP BY, ORDER BY

**Note:** Only orders that have at least one line item are counted.

In [ ]:
sql_q03 = """
SELECT s.store_id, s.city,
       COUNT(DISTINCT o.order_id) AS orders,
       SUM(oi.qty * oi.price)     AS revenue
FROM stores s
JOIN orders o       ON o.store_id = s.store_id
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY s.store_id, s.city
ORDER BY revenue DESC;
"""
df_q03 = run_query(sql_q03)
df_q03.head(10)

### Q04. Top products by revenue
**Question:** Find the top 10 products by total revenue and units sold.

**SQL concepts:** JOIN, SUM, GROUP BY, LIMIT

In [ ]:
sql_q04 = """
SELECT p.product_id,
       SUM(oi.qty)            AS units_sold,
       SUM(oi.qty * oi.price) AS revenue
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.product_id
ORDER BY revenue DESC
LIMIT 10;
"""
df_q04 = run_query(sql_q04)
df_q04.head(10)

### Q05. Category performance
**Question:** Compare units sold, revenue and number of orders across product categories.

**SQL concepts:** Multiple JOINs, GROUP BY, multiple aggregates

In [ ]:
sql_q05 = """
SELECT c.category_name,
       SUM(oi.qty)                  AS units_sold,
       SUM(oi.qty * oi.price)       AS revenue,
       COUNT(DISTINCT oi.order_id)  AS orders
FROM order_items oi
JOIN products p   ON p.product_id = oi.product_id
JOIN categories c ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY revenue DESC;
"""
df_q05 = run_query(sql_q05)
df_q05.head(10)

### Q06. Supplier revenue by country
**Question:** Find which supplier countries contribute the most revenue through their products.

**SQL concepts:** Multiple JOINs, SUM, GROUP BY

In [ ]:
sql_q06 = """
SELECT su.country,
       COUNT(DISTINCT su.supplier_id) AS suppliers,
       SUM(oi.qty * oi.price)         AS revenue
FROM order_items oi
JOIN products p   ON p.product_id = oi.product_id
JOIN suppliers su ON su.supplier_id = p.supplier_id
GROUP BY su.country
ORDER BY revenue DESC;
"""
df_q06 = run_query(sql_q06)
df_q06.head(10)

### Q07. Average order value by store city
**Question:** Calculate total revenue per order first, then the average order value for each store city.

**SQL concepts:** CTE, AVG, JOIN

In [ ]:
sql_q07 = """
WITH order_total AS (
    SELECT order_id, SUM(qty * price) AS total
    FROM order_items
    GROUP BY order_id
)
SELECT s.city,
       COUNT(*)                  AS orders,
       ROUND(AVG(ot.total), 2)   AS avg_order_value
FROM orders o
JOIN order_total ot ON ot.order_id = o.order_id
JOIN stores s       ON s.store_id = o.store_id
GROUP BY s.city
ORDER BY avg_order_value DESC;
"""
df_q07 = run_query(sql_q07)
df_q07.head(10)

### Q08. Monthly revenue trend
**Question:** Calculate monthly orders, units sold and revenue across the data period.

**SQL concepts:** DATE_TRUNC, GROUP BY

In [ ]:
sql_q08 = """
SELECT CAST(DATE_TRUNC('month', o.order_date) AS DATE) AS month,
       COUNT(DISTINCT o.order_id) AS orders,
       SUM(oi.qty)                AS units_sold,
       SUM(oi.qty * oi.price)     AS revenue
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY 1
ORDER BY 1;
"""
df_q08 = run_query(sql_q08)
df_q08.head(10)

### Q09. Year-over-year revenue growth
**Question:** Calculate yearly revenue and the percentage change from the previous year.

**SQL concepts:** CTE, LAG window function

**Note:** The first year has no previous year, so its growth is NULL (NaN).

In [ ]:
sql_q09 = """
WITH yearly AS (
    SELECT EXTRACT(YEAR FROM o.order_date) AS year,
           SUM(oi.qty * oi.price)          AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY 1
)
SELECT year,
       revenue,
       LAG(revenue) OVER (ORDER BY year) AS prev_year_revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY year))
             / LAG(revenue) OVER (ORDER BY year), 2) AS yoy_growth_pct
FROM yearly
ORDER BY year;
"""
df_q09 = run_query(sql_q09)
df_q09.head(10)

### Q10. Promotion vs non-promotion orders
**Question:** Compare order count, revenue and average order value for orders with and without a promotion.

**SQL concepts:** LEFT JOIN, CASE, GROUP BY

**Note:** Every order in this dataset has a promotion, so only the 'With promotion' group appears. The query still works if NULL promotions are added later.

In [ ]:
sql_q10 = """
WITH order_total AS (
    SELECT order_id, SUM(qty * price) AS total
    FROM order_items
    GROUP BY order_id
)
SELECT CASE WHEN p.promotion_id IS NULL THEN 'No promotion' ELSE 'With promotion' END AS promo_group,
       COUNT(*)                AS orders,
       SUM(ot.total)           AS revenue,
       ROUND(AVG(ot.total), 2) AS avg_order_value
FROM orders o
JOIN order_total ot    ON ot.order_id = o.order_id
LEFT JOIN promotions p ON p.promotion_id = o.promotion_id
GROUP BY 1
ORDER BY 1;
"""
df_q10 = run_query(sql_q10)
df_q10.head(10)

### Q11. Revenue by discount level
**Question:** Group orders by promotion discount value and compare order volume and revenue.

**SQL concepts:** JOIN, GROUP BY, AVG, SUM

In [ ]:
sql_q11 = """
WITH order_total AS (
    SELECT order_id, SUM(qty * price) AS total
    FROM order_items
    GROUP BY order_id
)
SELECT p.discount,
       COUNT(*)                AS orders,
       SUM(ot.total)           AS revenue,
       ROUND(AVG(ot.total), 2) AS avg_order_value
FROM orders o
JOIN order_total ot ON ot.order_id = o.order_id
JOIN promotions p   ON p.promotion_id = o.promotion_id
GROUP BY p.discount
ORDER BY p.discount;
"""
df_q11 = run_query(sql_q11)
df_q11.head(10)

### Q12. Return rate by category
**Question:** Calculate the share of order lines that were returned for each product category.

**SQL concepts:** JOIN, COUNT, LEFT JOIN, conditional counting

**Note:** 731 order lines have more than one return row, so returns are de-duplicated per line before counting.

In [ ]:
sql_q12 = """
WITH returned AS (
    SELECT DISTINCT order_item_id FROM returns
)
SELECT c.category_name,
       COUNT(*)                  AS order_lines,
       COUNT(rt.order_item_id)   AS returned_lines,
       ROUND(100.0 * COUNT(rt.order_item_id) / COUNT(*), 2) AS return_rate_pct
FROM order_items oi
JOIN products p   ON p.product_id = oi.product_id
JOIN categories c ON c.category_id = p.category_id
LEFT JOIN returned rt ON rt.order_item_id = oi.order_item_id
GROUP BY c.category_name
ORDER BY return_rate_pct DESC;
"""
df_q12 = run_query(sql_q12)
df_q12.head(10)

### Q13. Refunds by supplier country
**Question:** Find total refund value and number of returns linked to each supplier country.

**SQL concepts:** Multi-table JOIN, SUM, COUNT

In [ ]:
sql_q13 = """
SELECT su.country,
       COUNT(*)        AS returns,
       SUM(r.refund)   AS total_refund
FROM returns r
JOIN order_items oi ON oi.order_item_id = r.order_item_id
JOIN products p     ON p.product_id = oi.product_id
JOIN suppliers su   ON su.supplier_id = p.supplier_id
GROUP BY su.country
ORDER BY total_refund DESC;
"""
df_q13 = run_query(sql_q13)
df_q13.head(10)

### Q14. Shipment status distribution
**Question:** Count shipments by status and show each status as a percentage of all shipments.

**SQL concepts:** GROUP BY, COUNT, window percentage

In [ ]:
sql_q14 = """
SELECT status,
       COUNT(*) AS shipments,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM shipments
GROUP BY status
ORDER BY shipments DESC;
"""
df_q14 = run_query(sql_q14)
df_q14.head(10)

### Q15. Orders with missing shipment or payment
**Question:** Find orders that have no shipment record, or no payment record.

**SQL concepts:** LEFT JOIN, IS NULL (anti-join)

**Note:** Milestone 2 found exactly one payment and one shipment per order, so this is expected to return 0 rows (a clean result). The summary below confirms it.

In [ ]:
sql_q15 = """
SELECT o.order_id, o.order_date,
       CASE WHEN s.shipment_id IS NULL THEN 'missing' ELSE 'ok' END AS shipment,
       CASE WHEN p.payment_id  IS NULL THEN 'missing' ELSE 'ok' END AS payment
FROM orders o
LEFT JOIN shipments s ON s.order_id = o.order_id
LEFT JOIN payments  p ON p.order_id = o.order_id
WHERE s.shipment_id IS NULL OR p.payment_id IS NULL
ORDER BY o.order_id;
"""
df_q15 = run_query(sql_q15)
df_q15.head(10)

### Q15b. Q15 summary counts
**Question:** Summary of order-lifecycle completeness.

**SQL concepts:** Conditional aggregation

In [ ]:
sql_q15b = """
SELECT COUNT(*) AS total_orders,
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM shipments s WHERE s.order_id = o.order_id)) AS no_shipment,
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM payments p  WHERE p.order_id = o.order_id)) AS no_payment,
       COUNT(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM order_items i WHERE i.order_id = o.order_id)) AS no_items
FROM orders o;
"""
df_q15b = run_query(sql_q15b)
df_q15b.head(10)

### Q16. Payment vs order total
**Question:** Compare payments.amount with the line-item total of each order and list the largest differences.

**SQL concepts:** CTE, JOIN, ABS, ORDER BY

**Note:** Milestone 2 found that only 10 of 259,233 orders have a payment equal to the item total, so large differences are expected. This is why revenue is defined from order_items.

In [ ]:
sql_q16 = """
WITH item_total AS (
    SELECT order_id, SUM(qty * price) AS items_total
    FROM order_items
    GROUP BY order_id
), paid AS (
    SELECT order_id, SUM(amount) AS paid_total
    FROM payments
    GROUP BY order_id
)
SELECT it.order_id, it.items_total, pd.paid_total,
       ABS(pd.paid_total - it.items_total) AS abs_difference
FROM item_total it
JOIN paid pd ON pd.order_id = it.order_id
ORDER BY abs_difference DESC
LIMIT 100;
"""
df_q16 = run_query(sql_q16)
df_q16.head(10)

### Q17. Repeat customers
**Question:** Identify customers with more than one order and calculate the repeat-customer rate.

**SQL concepts:** COUNT, GROUP BY, HAVING

In [ ]:
sql_q17 = """
SELECT customer_id, COUNT(*) AS orders
FROM orders
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY orders DESC, customer_id
LIMIT 100;
"""
df_q17 = run_query(sql_q17)
df_q17.head(10)

### Q17b. Repeat-customer rate
**Question:** Share of ordering customers who placed more than one order.

**SQL concepts:** CTE, conditional counting

In [ ]:
sql_q17b = """
WITH per_customer AS (
    SELECT customer_id, COUNT(*) AS n
    FROM orders
    GROUP BY customer_id
)
SELECT COUNT(*) AS customers_with_orders,
       COUNT(*) FILTER (WHERE n > 1) AS repeat_customers,
       ROUND(100.0 * COUNT(*) FILTER (WHERE n > 1) / COUNT(*), 2) AS repeat_rate_pct
FROM per_customer;
"""
df_q17b = run_query(sql_q17b)
df_q17b.head(10)

### Q18. Customer lifetime value
**Question:** Find the top 10 customers by total spend and show their order counts.

**SQL concepts:** JOIN, SUM, GROUP BY, LIMIT

In [ ]:
sql_q18 = """
SELECT o.customer_id,
       COUNT(DISTINCT o.order_id) AS orders,
       SUM(oi.qty * oi.price)     AS total_spend
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY o.customer_id
ORDER BY total_spend DESC
LIMIT 10;
"""
df_q18 = run_query(sql_q18)
df_q18.head(10)

### Q19. Customers who never ordered
**Question:** List customers with no orders and summarise them by city and sign-up year.

**SQL concepts:** LEFT JOIN, IS NULL, GROUP BY

**Note:** If no customers qualify, the result is empty (all 50,000 customers placed at least one order).

In [ ]:
sql_q19 = """
SELECT c.city,
       EXTRACT(YEAR FROM c.signup_date) AS signup_year,
       COUNT(*) AS customers_without_orders
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
GROUP BY c.city, EXTRACT(YEAR FROM c.signup_date)
ORDER BY c.city, signup_year;
"""
df_q19 = run_query(sql_q19)
df_q19.head(10)

### Q20. Top customers within each city
**Question:** Rank customers by total spend within each city and keep the top 3 per city.

**SQL concepts:** Window function, PARTITION BY, RANK

In [ ]:
sql_q20 = """
WITH spend AS (
    SELECT c.customer_id, c.city, SUM(oi.qty * oi.price) AS total_spend
    FROM customers c
    JOIN orders o       ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY c.customer_id, c.city
), ranked AS (
    SELECT *, RANK() OVER (PARTITION BY city ORDER BY total_spend DESC) AS rnk
    FROM spend
)
SELECT city, rnk, customer_id, total_spend
FROM ranked
WHERE rnk <= 3
ORDER BY city, rnk;
"""
df_q20 = run_query(sql_q20)
df_q20.head(10)

### Q21. Top products within each category
**Question:** Rank products within each category by revenue and keep the top 3 per category.

**SQL concepts:** DENSE_RANK, PARTITION BY

In [ ]:
sql_q21 = """
WITH prod_rev AS (
    SELECT p.category_id, p.product_id, SUM(oi.qty * oi.price) AS revenue
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.category_id, p.product_id
), ranked AS (
    SELECT *, DENSE_RANK() OVER (PARTITION BY category_id ORDER BY revenue DESC) AS rnk
    FROM prod_rev
)
SELECT c.category_name, r.rnk, r.product_id, r.revenue
FROM ranked r
JOIN categories c ON c.category_id = r.category_id
WHERE r.rnk <= 3
ORDER BY c.category_name, r.rnk;
"""
df_q21 = run_query(sql_q21)
df_q21.head(10)

### Q22. Employees above store average salary
**Question:** List employees earning more than the average salary of their own store.

**SQL concepts:** Subquery/CTE, AVG, JOIN

In [ ]:
sql_q22 = """
WITH store_avg AS (
    SELECT store_id, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY store_id
)
SELECT e.employee_id, e.store_id, e.salary,
       ROUND(sa.avg_salary, 2) AS store_avg_salary
FROM employees e
JOIN store_avg sa ON sa.store_id = e.store_id
WHERE e.salary > sa.avg_salary
ORDER BY e.store_id, e.salary DESC;
"""
df_q22 = run_query(sql_q22)
df_q22.head(10)

### Q23. Revenue per employee by store
**Question:** Compare each store's revenue to its employee count and salary cost.

**SQL concepts:** CTE, multiple aggregates, JOIN

**Note:** Revenue and employees are aggregated in separate CTEs first, so the joins do not inflate the totals. Stores with no employees are excluded by the inner join.

In [ ]:
sql_q23 = """
WITH store_rev AS (
    SELECT o.store_id, SUM(oi.qty * oi.price) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY o.store_id
), store_emp AS (
    SELECT store_id, COUNT(*) AS employees, SUM(salary) AS salary_cost
    FROM employees
    GROUP BY store_id
)
SELECT s.store_id, s.city, sr.revenue, se.employees, se.salary_cost,
       ROUND(sr.revenue / se.employees, 2) AS revenue_per_employee
FROM stores s
JOIN store_rev sr ON sr.store_id = s.store_id
JOIN store_emp se ON se.store_id = s.store_id
ORDER BY revenue_per_employee DESC;
"""
df_q23 = run_query(sql_q23)
df_q23.head(10)

### Q24. Time from sign-up to first order
**Question:** For each customer calculate the days between signup_date and the first order, then summarise the average by city.

**SQL concepts:** MIN, date arithmetic, GROUP BY

**Note:** Milestone 2 found that 40% of orders are dated before the customer's sign-up date, so negative gaps are a data-quality issue in the source data, not a query error.

In [ ]:
sql_q24 = """
WITH first_order AS (
    SELECT customer_id, MIN(order_date) AS first_order_date
    FROM orders
    GROUP BY customer_id
), gap AS (
    SELECT c.customer_id, c.city, (f.first_order_date - c.signup_date) AS days_to_first_order
    FROM customers c
    JOIN first_order f ON f.customer_id = c.customer_id
)
SELECT city,
       COUNT(*)                                          AS customers,
       ROUND(AVG(days_to_first_order), 1)                AS avg_days,
       MIN(days_to_first_order)                          AS min_days,
       MAX(days_to_first_order)                          AS max_days,
       COUNT(*) FILTER (WHERE days_to_first_order < 0)   AS first_order_before_signup
FROM gap
GROUP BY city
ORDER BY avg_days;
"""
df_q24 = run_query(sql_q24)
df_q24.head(10)

## 6. Close the Connection

In [ ]:
engine.dispose()
print("Connection closed.")

## 7. Summary
All 24 analytical queries from Milestone 1 (plus two small helper queries, Q15b and Q17b) were run on the PostgreSQL server through `pandas.read_sql_query`. Each result was collected in a DataFrame and its first 10 rows printed with `head(10)`.

The queries use JOINs across up to four tables, GROUP BY with HAVING, CTEs, window functions (RANK, DENSE_RANK, LAG, SUM OVER), anti-joins, subqueries, conditional aggregation and date arithmetic.

Where the Milestone 2 data-quality findings matter (orders without items, payments unrelated to item totals, orders before sign-up, all orders having a promotion), the notes under each query explain how the result should be read.